# 🎓 Student Information

**Name:** Adarsh A  
**Roll No:** 23BSCED007  
**Course:** BSc (Cybersecurity, Ethical Hacking & Data Analytics)  
**Date:**   

---

# 🔐 Behavioral Analytics of Energy Security
## Analyzing User Login Patterns to Detect Suspicious Timing

---

| Field | Details |
|---|---|
| **Project Name** | Behavioral Analytics of Energy Security |
| **Objective** | Analyze user login patterns in an energy security system to find suspicious timing |
| **Anomaly Rule** | Flag any login that happens at 3 AM as suspicious |
| **Tools** | Jupyter Notebook, Python, Pandas |
| **Key Concept** | Anomaly Detection using Statistical Heuristics |

---

### 📌 What is Behavioral Analytics in Energy Security?

Energy security systems protect **critical infrastructure** such as power grids, substations, and SCADA (Supervisory Control and Data Acquisition) systems from unauthorized access and cyber threats.

**Behavioral analytics** monitors *how* and *when* authorized users access these systems and looks for **outliers** — patterns that deviate from normal behavior.

In this project, we analyze user login timestamps from an energy security platform.  
A login at **3 AM** is statistically rare for legitimate users and is flagged as a **potential security threat** — it could indicate:
- 🔴 Unauthorized access to energy control systems
- 🔴 Compromised user credentials
- 🔴 Insider threat activity
- 🔴 Automated bot or malware activity

---

## 📦 Step 1: Import Libraries

We import **Pandas** — the core Python library for data analysis.  
It allows us to load, clean, and analyze tabular data efficiently.

In [12]:
import pandas as pd

print('✅ Pandas version:', pd.__version__)
print('✅ Libraries imported successfully!')

✅ Pandas version: 2.3.3
✅ Libraries imported successfully!


## 📂 Step 2: Load the Dataset

We load the CSV file containing energy security system login records.  
Each row represents **one login event** by an authorized user into the energy security platform.

The `parse_dates` parameter automatically converts the `login_time` column into proper datetime format so we can extract hours, dates, etc.

In [13]:
df = pd.read_csv('user_login_data.csv', parse_dates=['login_time'])

print('✅ Energy Security Login Dataset Loaded!')
print(f'   Total login records  : {df.shape[0]}')
print(f'   Total columns        : {df.shape[1]}')
print(f'   Unique users         : {df["user_id"].nunique()}')
print(f'   Date range           : {df["login_time"].min()} to {df["login_time"].max()}')
print()
df.head(10)

✅ Energy Security Login Dataset Loaded!
   Total login records  : 30
   Total columns        : 5
   Unique users         : 10
   Date range           : 2024-01-15 03:02:00 to 2024-01-20 08:20:00



,user_id,username,login_time,department,location
0,U001,alice,2024-01-15 08:32:00,Engineering,Office
1,U002,bob,2024-01-15 03:14:00,Finance,Remote
2,U003,carol,2024-01-15 09:05:00,Engineering,Office
3,U004,dave,2024-01-15 03:02:00,IT,Remote
4,U005,eve,2024-01-15 10:45:00,HR,Office
5,U001,alice,2024-01-16 08:15:00,Engineering,Office
6,U002,bob,2024-01-16 11:30:00,Finance,Remote
7,U003,carol,2024-01-16 03:47:00,Engineering,Remote
8,U006,frank,2024-01-16 09:20:00,Operations,Office
9,U007,grace,2024-01-16 03:05:00,IT,Remote


## 🔍 Step 3: Explore the Data

Before performing any security analysis, we must understand the structure of our dataset.  
This step helps us verify data quality and identify any issues before analysis.

In [14]:
print('=== Dataset Structure ===')
df.info()

=== Dataset Structure ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   user_id     30 non-null     object        
 1   username    30 non-null     object        
 2   login_time  30 non-null     datetime64[ns]
 3   department  30 non-null     object        
 4   location    30 non-null     object        
dtypes: datetime64[ns](1), object(4)
memory usage: 1.3+ KB


In [15]:
print('=== Missing Values Check ===')
print(df.isnull().sum())
print()
print('=== Energy Security System Users ===')
print(df[['user_id','username','department']].drop_duplicates().reset_index(drop=True))

=== Missing Values Check ===
user_id       0
username      0
login_time    0
department    0
location      0
dtype: int64

=== Energy Security System Users ===
  user_id username   department
0    U001    alice  Engineering
1    U002      bob      Finance
2    U003    carol  Engineering
3    U004     dave           IT
4    U005      eve           HR
5    U006    frank   Operations
6    U007    grace           IT
7    U008    henry      Finance
8    U009     iris  Engineering
9    U010     jack           IT


## ⚙️ Step 4: Feature Engineering — Extract Login Hour

We extract the **hour of login** from the `login_time` timestamp using `.dt.hour`.

This gives us a number from **0 to 23** representing the hour of day.

> **Example:** `2024-01-15 03:14:00` → `login_hour = 3`  
> A login at hour **3** (3 AM) on an energy security system is highly suspicious.

In real-world energy security systems like **SCADA** or **power grid control panels**, access at odd hours is a major red flag used by security teams to trigger alerts.

In [16]:
df['login_hour'] = df['login_time'].dt.hour
df['login_date'] = df['login_time'].dt.date

print('✅ Login hour extracted successfully!')
print()
print('Login Hour Distribution (how many logins per hour):')
print(df['login_hour'].value_counts().sort_index())
print()
print('Sample — Login Hour Extraction:')
df[['username', 'login_time', 'login_hour']].head(10)

✅ Login hour extracted successfully!

Login Hour Distribution (how many logins per hour):
login_hour
3     12
7      2
8      6
9      4
10     2
11     2
13     1
14     1
Name: count, dtype: int64

Sample — Login Hour Extraction:


,username,login_time,login_hour
0,alice,2024-01-15 08:32:00,8
1,bob,2024-01-15 03:14:00,3
2,carol,2024-01-15 09:05:00,9
3,dave,2024-01-15 03:02:00,3
4,eve,2024-01-15 10:45:00,10
5,alice,2024-01-16 08:15:00,8
6,bob,2024-01-16 11:30:00,11
7,carol,2024-01-16 03:47:00,3
8,frank,2024-01-16 09:20:00,9
9,grace,2024-01-16 03:05:00,3


## 📊 Step 5: Calculate Average Login Hour Per User

We use `groupby()` to group all logins by user and calculate key statistics:

| Metric | Description |
|---|---|
| `avg_login_hour` | The typical hour this user accesses the energy security system |
| `total_logins` | Total number of login events |
| `min_hour` | Earliest login hour recorded |
| `max_hour` | Latest login hour recorded |
| `count_3am` | Number of 3 AM logins — the suspicious ones |

A legitimate energy security employee typically logs in during **office hours (8 AM – 6 PM)**.  
Any user with a very low or unusual average login hour warrants investigation.

In [17]:
user_stats = df.groupby(['user_id', 'username', 'department']).agg(
    total_logins   = ('login_time', 'count'),
    avg_login_hour = ('login_hour', 'mean'),
    min_hour       = ('login_hour', 'min'),
    max_hour       = ('login_hour', 'max'),
    count_3am      = ('login_hour', lambda x: (x == 3).sum())
).reset_index()

user_stats['avg_login_hour'] = user_stats['avg_login_hour'].round(2)

print('✅ User behavior statistics calculated!')
print()
user_stats

✅ User behavior statistics calculated!



,user_id,username,department,total_logins,avg_login_hour,min_hour,max_hour,count_3am
0,U001,alice,Engineering,5,7.80,7,8,0
1,U002,bob,Finance,4,5.00,3,11,3
2,U003,carol,Engineering,3,7.67,3,11,1
3,U004,dave,IT,3,6.67,3,14,2
4,U005,eve,HR,3,8.67,8,10,0
5,U006,frank,Operations,3,10.67,9,13,0
6,U007,grace,IT,3,3.00,3,3,3
7,U008,henry,Finance,2,5.00,3,7,1
8,U009,iris,Engineering,2,9.00,9,9,0
9,U010,jack,IT,2,3.00,3,3,2


## 🚨 Step 6: Flag Suspicious Logins (Anomaly Detection)

### The Security Heuristic Rule:
> **If `login_hour == 3` → Mark as SUSPICIOUS**

This is the core of **Anomaly Detection** — a rule-based approach widely used in:
- 🏭 Energy grid security monitoring
- 🔒 SCADA system protection
- 🛡️ SIEM (Security Information and Event Management) platforms
- 🔍 Insider threat detection programs

When a user accesses an energy security system at 3 AM without prior authorization or a scheduled maintenance window, it triggers a **security alert** for investigation.

In [18]:
df['is_suspicious'] = df['login_hour'] == 3

total      = len(df)
suspicious = df['is_suspicious'].sum()
normal     = total - suspicious

print('=======================================')
print('  ENERGY SECURITY — ANOMALY REPORT')
print('=======================================')
print(f'  Total login events   : {total}')
print(f'  Normal logins        : {normal}')
print(f'  Suspicious (3 AM)    : {suspicious}')
print(f'  Anomaly rate         : {round(suspicious/total*100, 1)}%')
print('=======================================')

  ENERGY SECURITY — ANOMALY REPORT
  Total login events   : 30
  Normal logins        : 18
  Suspicious (3 AM)    : 12
  Anomaly rate         : 40.0%


In [19]:
suspicious_logins = df[df['is_suspicious'] == True][[
    'user_id', 'username', 'department', 'login_time', 'login_hour', 'location'
]].reset_index(drop=True)

suspicious_logins['flag_reason'] = 'Unauthorized 3 AM access to Energy Security System'
suspicious_logins['risk_level']  = 'HIGH'

print(f'🚨 {len(suspicious_logins)} suspicious login(s) detected on Energy Security System:\n')
suspicious_logins

🚨 12 suspicious login(s) detected on Energy Security System:



,user_id,username,department,login_time,login_hour,location,flag_reason,risk_level
0,U002,bob,Finance,2024-01-15 03:14:00,3,Remote,Unauthorized 3 AM access to Energy Security Sy...,HIGH
1,U004,dave,IT,2024-01-15 03:02:00,3,Remote,Unauthorized 3 AM access to Energy Security Sy...,HIGH
2,U003,carol,Engineering,2024-01-16 03:47:00,3,Remote,Unauthorized 3 AM access to Energy Security Sy...,HIGH
3,U007,grace,IT,2024-01-16 03:05:00,3,Remote,Unauthorized 3 AM access to Energy Security Sy...,HIGH
4,U008,henry,Finance,2024-01-17 03:55:00,3,Remote,Unauthorized 3 AM access to Energy Security Sy...,HIGH
5,U002,bob,Finance,2024-01-17 03:10:00,3,Remote,Unauthorized 3 AM access to Energy Security Sy...,HIGH
6,U010,jack,IT,2024-01-18 03:33:00,3,Remote,Unauthorized 3 AM access to Energy Security Sy...,HIGH
7,U007,grace,IT,2024-01-18 03:22:00,3,Remote,Unauthorized 3 AM access to Energy Security Sy...,HIGH
8,U004,dave,IT,2024-01-19 03:01:00,3,Remote,Unauthorized 3 AM access to Energy Security Sy...,HIGH
9,U002,bob,Finance,2024-01-19 03:58:00,3,Remote,Unauthorized 3 AM access to Energy Security Sy...,HIGH


## 👤 Step 7: Identify Flagged vs Safe Users

We now separate users into two groups:
- 🚨 **Flagged users** — had at least one 3 AM login (require security investigation)
- ✅ **Safe users** — all logins within normal business hours

In [20]:
user_stats['flagged'] = user_stats['count_3am'] > 0

flagged_users = user_stats[user_stats['flagged'] == True].reset_index(drop=True)
safe_users    = user_stats[user_stats['flagged'] == False].reset_index(drop=True)

print(f'🚨 HIGH RISK users  : {len(flagged_users)} (require immediate investigation)')
print(f'✅ Safe users       : {len(safe_users)} (normal login behavior)')
print()
print('=== 🚨 Flagged Users — Energy Security Risk ===')
flagged_users[['user_id','username','department','total_logins','avg_login_hour','count_3am']]

🚨 HIGH RISK users  : 6 (require immediate investigation)
✅ Safe users       : 4 (normal login behavior)

=== 🚨 Flagged Users — Energy Security Risk ===


,user_id,username,department,total_logins,avg_login_hour,count_3am
0,U002,bob,Finance,4,5.00,3
1,U003,carol,Engineering,3,7.67,1
2,U004,dave,IT,3,6.67,2
3,U007,grace,IT,3,3.00,3
4,U008,henry,Finance,2,5.00,1
5,U010,jack,IT,2,3.00,2


In [21]:
print('=== ✅ Safe Users — Normal Behavior ===')
safe_users[['user_id','username','department','total_logins','avg_login_hour']]

=== ✅ Safe Users — Normal Behavior ===


,user_id,username,department,total_logins,avg_login_hour
0,U001,alice,Engineering,5,7.80
1,U005,eve,HR,3,8.67
2,U006,frank,Operations,3,10.67
3,U009,iris,Engineering,2,9.00


## 📋 Step 8: Complete Security Audit Log

This is the final **Security Audit Log** — a complete view of all login events with anomaly flags applied.  
In a real energy security operations center, this table would feed into a dashboard monitored 24/7.

In [22]:
df['status']     = df['is_suspicious'].apply(lambda x: '🚨 SUSPICIOUS' if x else '✅ Normal')
df['risk_level'] = df['is_suspicious'].apply(lambda x: 'HIGH' if x else 'LOW')

print('=== Complete Energy Security Audit Log ===')
df[['user_id','username','department','login_time','login_hour','status','risk_level']]

=== Complete Energy Security Audit Log ===


,user_id,username,department,login_time,login_hour,status,risk_level
0,U001,alice,Engineering,2024-01-15 08:32:00,8,✅ Normal,LOW
1,U002,bob,Finance,2024-01-15 03:14:00,3,🚨 SUSPICIOUS,HIGH
2,U003,carol,Engineering,2024-01-15 09:05:00,9,✅ Normal,LOW
3,U004,dave,IT,2024-01-15 03:02:00,3,🚨 SUSPICIOUS,HIGH
4,U005,eve,HR,2024-01-15 10:45:00,10,✅ Normal,LOW
5,U001,alice,Engineering,2024-01-16 08:15:00,8,✅ Normal,LOW
6,U002,bob,Finance,2024-01-16 11:30:00,11,✅ Normal,LOW
7,U003,carol,Engineering,2024-01-16 03:47:00,3,🚨 SUSPICIOUS,HIGH
8,U006,frank,Operations,2024-01-16 09:20:00,9,✅ Normal,LOW
9,U007,grace,IT,2024-01-16 03:05:00,3,🚨 SUSPICIOUS,HIGH


## 💾 Step 9: Export Security Reports

We export two CSV report files:
1. `anomaly_report.csv` — only the suspicious 3 AM logins (for security team action)
2. `full_security_audit_log.csv` — complete audit log with all flags (for record keeping)

In [23]:
suspicious_logins.to_csv('anomaly_report.csv', index=False)
df.to_csv('full_security_audit_log.csv', index=False)

print('✅ anomaly_report.csv          — Suspicious logins exported')
print('✅ full_security_audit_log.csv — Complete audit log exported')
print()
print(f'Total flagged events saved : {len(suspicious_logins)}')
print(f'Total audit records saved  : {len(df)}')

✅ anomaly_report.csv          — Suspicious logins exported
✅ full_security_audit_log.csv — Complete audit log exported

Total flagged events saved : 12
Total audit records saved  : 30


---

## ✅ Conclusion

In this project we successfully built a **Behavioral Analytics system for Energy Security** using Python and Pandas.

| Step | What We Did | Python/Pandas Used |
|---|---|---|
| Step 1 | Imported libraries | `import pandas as pd` |
| Step 2 | Loaded login dataset | `pd.read_csv()` |
| Step 3 | Explored data quality | `df.info()`, `df.isnull()` |
| Step 4 | Extracted login hour | `df['login_time'].dt.hour` |
| Step 5 | Calculated user averages | `df.groupby()`, `.agg()`, `.mean()` |
| Step 6 | Detected 3 AM anomalies | Boolean masking `df['login_hour'] == 3` |
| Step 7 | Identified risky users | Filtering with conditions |
| Step 8 | Built security audit log | `.apply()`, `lambda` |
| Step 9 | Exported security reports | `df.to_csv()` |

---

### 🧠 Key Takeaway

> This **statistical heuristic** (flag login_hour == 3) is the foundation of real-world anomaly detection used in **energy grid cybersecurity**, SCADA protection, and SIEM platforms. Behavioral analytics helps security teams catch threats early — before they cause damage to critical energy infrastructure.

---
**Project:** Behavioral Analytics of Energy Security  
**Tools:** Python · Pandas · Jupyter Notebook  
**Concept:** Anomaly Detection · Statistical Heuristics · Behavioral Analytics